## Match two datasheets using simple rule-based approach

### Target data
Large databump from e.g. Wikidata or Getty ULAN containing basic biographical information like fullname, gender, years and places of birth and death and identifiers in external datasources

### Source data
JSON formatted data like the Echolot examples available at repository [echolot-WP5](https://github.com/ECHOLOT-ECCCH/echolot-WP5/tree/main). 

### Implementation of rule-based approach


## Import modules and libraries

In [ ]:
import csv
import json
import re
import unicodedata

import fuzzy
import textdistance

from collections import defaultdict
from dataclasses import dataclass, field
from functools import lru_cache
from itertools import islice
from math import isclose
from typing import Optional

from ruleBasedMatcher import RuleBasedMatcher

In [ ]:
TARGETFILE = 'datasheets/wikidata.tsv'
OUTFILE = 'datasheets/test.tsv'

In [ ]:
matcher = RuleBasedMatcher(target_file='datasheets/wikidata.tsv')

In [ ]:
# match_record ignored extra parameters like 'floruit_start' in this example
matcher.match_record(id='http://example.com/ID123456789', birth_year=1951, floruit_start=1975, fullname='Edwin Aafjes')

In [ ]:
# source_file = 'datasheets/ulan.tsv'
source_file = '../../echolot-WP5/5.1 Basque CH/Inguma/2026_may_58_not_on_Wikidata_v2.json'
source_file = '../../echolot-WP5/5.1 Basque CH/TBK/authors_for_recon.json'
matcher.year_tolerance = 0
matches = matcher.match_file(source_file=source_file)

# Found 1382 matches out of 10988 entries (12.58%).
print(f'Found in total {len(matches)} matches.')

In [ ]:
for key, match in islice(matches.items(), 12):
    print(f'{key}\n\t{match}')
# Found in total 3235 matches

In [ ]:
# The datasheet EcholotToWikidata.tsv is queried from
# ehkultura.wikibase.cloud
# https://tinyurl.com/22uynzgh
infile = 'datasheets/EcholotToWikidata.tsv'
echolot_to_wikidata_lookup = {}
with open(infile, newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile, delimiter='\t')
    echolot_to_wikidata_lookup = {row['id']: row['wikidata'] for row in reader}

print(f'{len(echolot_to_wikidata_lookup)} links found.')
for ob in islice(echolot_to_wikidata_lookup.items(), 5):
    print(ob)

In [ ]:
tp, fp = 0,0
for key, arr in islice(matches.items(), None):
    for score, match in arr:
        if echolot_wikidata := echolot_to_wikidata_lookup.get(key.id):
            if echolot_wikidata == match.wikidata:
                # print(key.id, score, match, echolot_wikidata)
                tp += 1
            else:
                fp += 1
"""
Linked 3235 records out of total 8451 (38.28%)
Precision for matches 2779/2833 matches (98.09%)
"""

print(f'Linked {len(matches)} records out of total {len(echolot_to_wikidata_lookup)} ({len(matches)/len(echolot_to_wikidata_lookup):.2%})')
print(f'Precision for matches {tp}/{tp+fp} matches ({tp/(tp+fp or 1):.2%}).')

In [ ]:
matcher.year_tolerance = 1
matches_1 = matcher.match_file(source_file=source_file)

i = 0
for match, arr in matches_1.items():
    if match not in matches:
        print(f'{match} -> {arr}')
        i += 1
print(f'In total {i} possible new matches found with year_tolerance=1.')

## TODO add output to (decided CSV/JSON/?) format